In [0]:
import os
from pyspark.sql.functions import col, current_timestamp, current_date

# 1. Parameter Collection from Widgets
_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = dbutils.widgets.get("secret_scope")

# Target Definitions
catalog_name = "dbr_dev"
login_prefix = "joshuandegwa"
schema_name = f"{login_prefix}_bronze"
volume_name = "raw_files"
dataset_folder = "banking"

# Base Paths
volume_base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{dataset_folder}"
storage_base_url = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"

# Ensure active SQL context
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

# 2. Find CSV files to extract entity names
landing_files = [f for f in os.listdir(volume_base_path) if f.endswith(".csv")]

print(f"Found {len(landing_files)} CSV files: {landing_files}\n")

# 3. Dynamic Ingestion Loop using pathGlobFilter
for file in landing_files:
    entity_name = os.path.splitext(file)[0]  # e.g., "accounts", "loans"
    target_table_fqn = f"`{catalog_name}`.`{schema_name}`.`bronze_{entity_name}`"
    
    checkpoint_path = f"{storage_base_url}/_checkpoints/bronze/{entity_name}"
    schema_location = f"{storage_base_url}/_schemas/bronze/{entity_name}"
    
    print(f" Ingesting: {file} -> {target_table_fqn}")

    df_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", file)  # Filter for this specific CSV inside the directory
        .option("header", "true")
        .load(volume_base_path)          # Point to DIRECTORY, not file
        .withColumn("_source_filename", col("_metadata.file_path"))
        .withColumn("_ingested_timestamp", current_timestamp())
        .withColumn("_load_date", current_date())
    )

    query = (
        df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table_fqn)
    )

    query.awaitTermination()
    print(f" Successfully written to {target_table_fqn}\n")

print(" Bronze ingestion complete!")